# GLM-5.2 transient activation memory per layer

How much memory one decoder layer's backward needs, and how it splits between attention and MoE.

Per-token coefficients are measured from the B300 CP8/EP8 allocator snapshots under `../runs/`,
then scaled to any `S`, `CP`, `EP`.

In [1]:
import pickle
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

HIDDEN = 6_144
EXPERT_FFN = 2_048
TOP_K = 8
NUM_LAYERS = 78
NUM_EXPERTS = 256
BF16 = 2  # bytes
GIB = 2**30

# B300, TP1/PP1/CP8/EP8, LoRA r32, native-FP8 experts, HybridEP, full one-layer recompute.
PROFILE_ROOT = Path("../runs/glm52_b200_b300_rightsizing_20260901/glm52/b300/pp1_cp8_ep8")
PROFILE_CP = 8
PROFILE_EP = 8
PROFILES = {
    131_072: PROFILE_ROOT / "131k/memory/memory.rank0.pickle",
    262_144: PROFILE_ROOT / "262k/memory/memory.rank0.pickle",
}

## What full recompute holds

Trainer default is `recompute.granularity="full"`, uniform, one layer per checkpoint.

- Forward keeps only each layer's input residual (`[S/CP, 6144]` bf16). All 78 are live at the LM-head peak.
- Backward re-runs one layer's forward, which materialises every saved-for-backward tensor of that layer
  at once, then frees each as its backward op consumes it.
- Two moments compete for the layer peak: the end of the recompute forward (MoE dominates) and the
  DSA attention backward (MoE already freed; attention workspace is ~3.5x the attention saved set).

The residual frees mark layer boundaries: layer `i`'s output is the checkpointed input of layer `i+1`
and is freed when layer `i+1` finishes its backward.

In [2]:
def load_events(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(path.resolve())
    snapshot = pickle.load(path.open("rb"))
    rank = int(path.stem.removeprefix("memory.rank"))
    return snapshot["device_traces"][rank]


def replay(events):
    """Yield (index, live_bytes, live_tensors_by_addr) after each event."""
    live, total = {}, 0
    for index, event in enumerate(events):
        action = event.get("action")
        if action == "alloc":
            total += event["size"]
            live[event["addr"]] = event
        elif action == "free_requested":
            total -= event["size"]
            live.pop(event["addr"], None)
        yield index, total, live


def site(event) -> str:
    frame = (event.get("frames") or [{}])[0]
    return f"{frame.get('name')}@{Path(frame.get('filename', '')).name}:{frame.get('line')}"


def category(event) -> str:
    stack = " ".join(
        f"{f.get('name', '')} {f.get('filename', '')}" for f in event.get("frames") or []
    ).lower()
    if "_dequantize_fp8_weights_to_bf16" in stack:
        return "expert_dequant"
    if any(key in stack for key in ("expert", "moe", "grouped")):
        return "moe"
    if any(key in stack for key in ("attention", "indexer", "dsa", "mla")):
        return "attention"
    if "rmsnorm" in stack or "layernorm" in stack:
        return "norm"
    return "other"

In [3]:
RESIDUAL_SITE = "_bias_dropout_add_func"
DISPATCH_SITE = "dispatch_with_permute@hybrid_ep_buffer.py:419"


@dataclass
class LayerSegment:
    """One decoder layer's recompute-forward + backward."""

    growth: int  # peak live bytes above the layer's starting live set
    new_tensors: list[dict]  # live at that peak but not at the layer start

    def bytes_by_category(self) -> Counter:
        counts = Counter()
        for event in self.new_tensors:
            counts[category(event)] += event["size"]
        return counts

    @property
    def routed_rows(self) -> float:
        dispatched = sum(e["size"] for e in self.new_tensors if site(e) == DISPATCH_SITE)
        return dispatched / (HIDDEN * BF16)


def layer_segments(events) -> tuple[int, list[LayerSegment]]:
    """Return (LM-head peak growth, per-layer segments in backward order)."""
    peak_index, peak_bytes, live_at_peak = -1, -1, {}
    for index, total, live in replay(events):
        if total > peak_bytes:
            peak_index, peak_bytes, live_at_peak = index, total, dict(live)

    residuals = {addr for addr, e in live_at_peak.items() if site(e).startswith(RESIDUAL_SITE)}
    if len(residuals) != NUM_LAYERS:
        raise ValueError(f"expected {NUM_LAYERS} residuals live at the peak, found {len(residuals)}")

    # First free of each residual after the peak; the allocator recycles addresses.
    boundaries, pending = set(), set(residuals)
    for index in range(peak_index + 1, len(events)):
        event = events[index]
        if event.get("action") == "free_requested" and event["addr"] in pending:
            pending.remove(event["addr"])
            boundaries.add(index)

    segments = []
    start_bytes, start_live, seg_peak_bytes, seg_peak_live = None, None, None, None
    for index, total, live in replay(events):
        if index in boundaries:
            if start_live is not None:
                new = [e for addr, e in seg_peak_live.items() if addr not in start_live]
                segments.append(LayerSegment(seg_peak_bytes - start_bytes, new))
            start_bytes, start_live = total, dict(live)
            seg_peak_bytes, seg_peak_live = total, dict(live)
        elif start_live is not None and total > seg_peak_bytes:
            seg_peak_bytes, seg_peak_live = total, dict(live)
    return peak_bytes, segments

In [4]:
def describe(seqlen: int, path: Path) -> list[LayerSegment]:
    tokens_per_rank = seqlen // PROFILE_CP
    peak_bytes, layers = layer_segments(load_events(path))
    growths = sorted(layer.growth for layer in layers)
    print(
        f"{seqlen:,} tokens ({tokens_per_rank:,} per rank): LM-head peak {peak_bytes / GIB:.1f} GiB, "
        f"per-layer peak {growths[0] / GIB:.1f}-{growths[-1] / GIB:.1f} GiB"
    )
    for k, layer in enumerate(layers[:8]):
        cats = "  ".join(f"{c}={b / GIB:.2f}" for c, b in layer.bytes_by_category().most_common(3))
        print(f"  layer {NUM_LAYERS - 1 - k}: {layer.growth / GIB:5.2f} GiB  {cats}")
    return layers


layers_by_seqlen = {seqlen: describe(seqlen, path) for seqlen, path in PROFILES.items()}

131,072 tokens (16,384 per rank): LM-head peak 47.2 GiB, per-layer peak 12.2-18.9 GiB
  layer 77: 12.19 GiB  attention=10.80  other=1.38  moe=0.02
  layer 76: 13.04 GiB  moe=7.16  attention=3.44  expert_dequant=2.25
  layer 75: 12.19 GiB  attention=10.80  other=1.38  moe=0.02
  layer 74: 18.94 GiB  moe=13.06  attention=3.38  expert_dequant=2.25
  layer 73: 12.19 GiB  attention=10.80  other=1.38  moe=0.02
  layer 72: 14.54 GiB  moe=8.66  attention=3.44  expert_dequant=2.25
  layer 71: 12.19 GiB  attention=10.80  other=1.38  moe=0.02
  layer 70: 12.19 GiB  attention=10.80  other=1.38  moe=0.02
262,144 tokens (32,768 per rank): LM-head peak 88.5 GiB, per-layer peak 24.3-36.6 GiB
  layer 77: 24.39 GiB  attention=21.60  other=2.75  moe=0.04
  layer 76: 24.39 GiB  attention=21.60  other=2.75  moe=0.04
  layer 75: 24.39 GiB  attention=21.60  other=2.75  moe=0.04
  layer 74: 36.62 GiB  moe=27.14  attention=7.11  expert_dequant=2.25
  layer 73: 24.39 GiB  attention=21.60  other=2.75  moe=0.04
 

In [5]:
@dataclass
class PerTokenCoefficients:
    """Bytes per local token unless noted, at the two intra-layer peaks."""

    attention_saved: float  # live when the MoE backward starts
    norm_saved: float
    moe_saved_per_row: float  # per routed (token, expert) row
    expert_dequant_per_expert: float  # bf16 copy of one frozen FP8 expert, constant in S
    attention_backward: float  # whole-layer growth during the attention backward
    routing_imbalance: float  # measured rows / even share on this rank

    def show(self):
        for name, value in vars(self).items():
            text = f"{value:.2f}x" if name == "routing_imbalance" else f"{value:,.0f} B"
            print(f"  {name:28s} {text:>16s}")


def coefficients(layers: list[LayerSegment], tokens_per_rank: int) -> PerTokenCoefficients:
    moe_peak = max(layers, key=lambda layer: layer.bytes_by_category()["moe"])
    attention_peak = max(layers, key=lambda layer: layer.bytes_by_category()["attention"])
    at_moe_peak = moe_peak.bytes_by_category()
    even_rows = tokens_per_rank * PROFILE_CP * TOP_K / PROFILE_EP
    return PerTokenCoefficients(
        attention_saved=at_moe_peak["attention"] / tokens_per_rank,
        norm_saved=at_moe_peak["norm"] / tokens_per_rank,
        moe_saved_per_row=at_moe_peak["moe"] / moe_peak.routed_rows,
        expert_dequant_per_expert=at_moe_peak["expert_dequant"] / (NUM_EXPERTS / PROFILE_EP),
        attention_backward=attention_peak.growth / tokens_per_rank,
        routing_imbalance=moe_peak.routed_rows / even_rows,
    )


for seqlen, layers in layers_by_seqlen.items():
    print(f"{seqlen:,} tokens:")
    coefficients(layers, seqlen // PROFILE_CP).show()

coef = coefficients(layers_by_seqlen[131_072], 131_072 // PROFILE_CP)

131,072 tokens:
  attention_saved                     221,264 B
  norm_saved                           12,288 B
  moe_saved_per_row                    51,335 B
  expert_dequant_per_expert        75,497,472 B
  attention_backward                  799,076 B
  routing_imbalance                       2.08x
262,144 tokens:
  attention_saved                     233,040 B
  norm_saved                           12,292 B
  moe_saved_per_row                    51,248 B
  expert_dequant_per_expert        75,497,472 B
  attention_backward                  799,060 B
  routing_imbalance                       2.17x


## Full-layer model

`moe_bound` is the end of the recompute forward; `attention_bound` is the attention backward.
The layer peak is the larger of the two. At EP=1 routing is exact (`routing_imbalance=1`).

In [6]:
def full_layer_peak_bytes(S: int, coef: PerTokenCoefficients, cp=1, ep=1, routing_imbalance=1.0):
    tokens = S / cp
    rows = S * TOP_K / ep * routing_imbalance
    local_experts = NUM_EXPERTS / ep
    moe_bound = (
        tokens * (coef.attention_saved + coef.norm_saved)
        + rows * coef.moe_saved_per_row
        + local_experts * coef.expert_dequant_per_expert
    )
    attention_bound = tokens * coef.attention_backward
    return {
        "moe_bound": moe_bound,
        "attention_bound": attention_bound,
        "peak": max(moe_bound, attention_bound),
    }


S = 262_144/8

print(f"one layer, TP=CP=EP=1, S={S:,}")
for name, value in full_layer_peak_bytes(S, coef).items():
    print(f"  {name:16s} {value / 1e9:6.1f} GB")

print(f"\nsanity check at the profiled shape (measured layer peak 18.88 GiB)")
profiled = full_layer_peak_bytes(131_072, coef, cp=8, ep=8, routing_imbalance=coef.routing_imbalance)
for name, value in profiled.items():
    print(f"  {name:16s} {value / GIB:6.2f} GiB")

one layer, TP=CP=EP=1, S=32,768.0
  moe_bound          40.4 GB
  attention_bound    26.2 GB
  peak               40.4 GB

sanity check at the profiled shape (measured layer peak 18.88 GiB)
  moe_bound         18.88 GiB
  attention_bound   12.19 GiB
  peak              18.88 GiB


In [7]:
# Analytical routed-MoE saved set per row, live at the MoE peak:
# dispatched input [C, H], FC1 out [C, 2I], gate and up [C, I], SiLU [C, I], gate*up [C, I], FC2 out [C, H].
analytical_per_row = (2 * HIDDEN + 6 * EXPERT_FFN) * BF16
print(f"analytical {analytical_per_row:,} B/row, measured {coef.moe_saved_per_row:,.0f} B/row")
print("(router outputs, permutation indices and the shared expert make up the difference)")

analytical 49,152 B/row, measured 51,335 B/row
(router outputs, permutation indices and the shared expert make up the difference)
